# 08 - Follow-ups (short-term memory) and user facts (long-term memory)

**Scenario.** Kavya Reddy uses the support chat on two different days.

- **Short-term memory** (checkpointer, per `thread_id`): a follow-up like *"and how do I move it to a new phone?"*
  is understood from the conversation.
- **Long-term memory** (store, per `user_id`, owned by the Coordinator): facts such as her customer ID and phone model
  are saved after each answer and recalled on a **new thread**.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

In [ ]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

In [ ]:
USER = "cust-kavya"
print("forgot", await service.forget(USER), "old facts")   # start the demo from a clean slate

day1 = await service.ask("Hi, I'm Kavya Reddy, customer CUST-0003. How do I set up an eSIM on my phone?",
                         user_id=USER, on_event=print)
day1

## Follow-up on the same thread
"it" and "my phone" are resolved from the conversation history.

In [ ]:
follow_up = await service.ask("And how do I move it to a new phone later?", thread_id=day1.thread_id,
                             user_id=USER, on_event=print)
follow_up

## What the Coordinator remembered about Kavya

In [ ]:
for fact in await service.recall(USER):
    print("-", fact)

## A new day, a new thread
No conversation history - but the long-term facts are loaded by `load_memory`, so the planner already knows her phone model.

In [ ]:
day2 = await service.ask("Does my phone support wireless charging?", user_id=USER, on_event=print)
day2

In [ ]:
# Close the memory databases
await service.aclose()

## What to observe
- The follow-up needs no restatement of the phone or the topic.
- On the new thread, `load_memory` reports recalled facts and the plan targets the Aria 12 directly.
- `service.forget(user_id)` deletes the long-term facts (right to be forgotten).